# Capstone · Notebook 4. Ablations: from a hypothesis to a measurement

> **Files for this notebook.** Download [course-materials.zip](https://vladkulikov.com/downloads/modern-llms/course-materials.zip) from the course website and extract the complete archive, preserving the `capstone` folder. In Colab, upload the same ZIP when the notebook prompts for files.

*Modern LLMs course · final practical project*

The nineteen modules introduced mechanisms together with their assumptions. Here we ask narrower experimental questions: how normalization changes activation scale, how much KV memory GQA saves, what changes when experts replace an FFN, and how a model handles dependency lengths absent from training.

For each comparison, state the factor, controls, and measurement in advance. One conceptual change can require several coordinated config fields. A shared seed makes initialization reproducible, but cannot make every initial tensor identical across different architectures.

On a first pass, run the cells and interpret each table. On a second, select an experiment and repeat it across seeds to test a hypothesis. The final report is assembled from this run's results. Fast validation checks execution; it does not substitute for sufficient training.

## 1. Common setup

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for package_root in (search_root / "capstone" / "minillm" / "en", search_root / "minillm" / "en"):
        found = _activate_capstone_root(package_root)
        if found:
            return found
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "project package: " + found

    try:
        import minillm  # noqa: F401
        return "minillm is already importable"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB)) + sorted(Path.cwd().glob("course-materials.zip"))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Upload course-materials.zip from the course notebook page. The directory structure will be restored automatically.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB)) + sorted(Path.cwd().glob("course-materials.zip"))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        root = _find_capstone_root(destination)
        if root:
            return "project archive: " + str(archive_path)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "project archive: " + str(archive_path)

    raise RuntimeError("minillm was not found. Download course-materials.zip from the course notebook page and either extract it in full or upload the ZIP to the current Colab session.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import math, os, random
import torch, numpy as np
import matplotlib.pyplot as plt

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = 60 if DEVICE == "cpu" else 600
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    STEPS = 5     # on a GPU the contrasts are sharper
print("device:", DEVICE, "| steps per run:", STEPS)

In [ ]:
from minillm.config import MiniLLMConfig, kv_bytes_per_token
from minillm.data import TokenStream
from minillm.model import MiniLLM, RMSNorm
from minillm.tokenizer import BBPETokenizer
from minillm.train import TrainConfig, evaluate, train

def high_entropy_corpus(n_lines=8_000, lexicon=600, seed=0) -> str:
    """The same pseudo-language as notebook 3: a Zipfian lexicon leaves models
    the headroom in which architectural variants are distinguishable."""
    rng = random.Random(seed)
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    words = ["".join(rng.choice(alphabet) for _ in range(rng.randint(3, 8)))
             for _ in range(lexicon)]
    weights = [1.0 / (i + 1) ** 0.9 for i in range(lexicon)]
    return "".join(" ".join(rng.choices(words, weights=weights, k=rng.randint(6, 14)))
                   + ".\n" for _ in range(n_lines))

corpus = high_entropy_corpus(seed=SEED)
lines = corpus.splitlines(keepends=True)
a, b = int(len(lines) * .8), int(len(lines) * .9)
train_text, valid_text, test_text = map("".join, (lines[:a], lines[a:b], lines[b:]))
tokenizer = BBPETokenizer.train(train_text, vocab_size=1024, specials=["<|eos|>"])
train_stream = TokenStream.from_text(train_text, tokenizer, 128)
valid_stream = TokenStream.from_text(valid_text, tokenizer, 128)
test_stream = TokenStream.from_text(test_text, tokenizer, 128)
CEILING = math.log(tokenizer.vocab_size)
print(f"vocabulary {tokenizer.vocab_size} · train {len(train_stream):,} windows · "
      f"valid {len(valid_stream):,} windows · uniform reference ln(V) = {CEILING:.3f}")

In [ ]:
BASE = dict(vocab_size=tokenizer.vocab_size, hidden_size=128, num_hidden_layers=3,
            num_attention_heads=4, num_key_value_heads=2, head_dim=32,
            max_position_embeddings=256)

def run(label: str, steps: int = None, **overrides):
    """One ablation run: fixed seed, fixed data, fixed schedule."""
    cfg = MiniLLMConfig.profile("S", **{**BASE, **overrides})
    torch.manual_seed(SEED)
    model = MiniLLM(cfg).to(DEVICE)
    tcfg = TrainConfig(steps=steps or STEPS, seq_len=128, micro_batch_size=8,
                       optimizer="muon", seed=SEED, device=DEVICE, compute_dtype="auto", log_every=10_000)
    train(model, train_stream, tcfg)
    loss = evaluate(model, valid_stream, tcfg, batches=8)
    total = cfg.parameter_report()["total"]
    print(f"  {label:<26} parameters {total:>9,} · active "
          f"{cfg.active_parameters():>9,} · valid loss {loss:.4f}")
    return dict(label=label, cfg=cfg, model=model, loss=loss, params=total,
                active=cfg.active_parameters())

## 2. Norm placement (Module 6)

Compare pre-norm, post-norm, Peri-LN, OLMo2-style normalization, and a no-norm control. These normalize different parts of the computation graph; sharing a scale-related purpose does not make them equivalent.

Start with a separate residual chain $h\leftarrow h+\tanh(h)W$, using random matrix entries with variance $1/d$. This is a nonlinear illustration rather than the lecture's exact linear recurrence. Trace RMS over 36 layers. The bounded tanh changes growth behavior, so exponential growth is not assumed.

In [ ]:
torch.manual_seed(1)
d, layers = 256, 36
h0 = torch.randn(1, d)
weights = [torch.randn(d, d) / math.sqrt(d) for _ in range(layers)]
norm = RMSNorm(d, 1e-6)

traces = {}
with torch.no_grad():
    for mode in ("none", "pre", "post"):
        h, trace = h0.clone(), []
        for W in weights:
            if mode == "none":
                h = h + torch.tanh(h) @ W
            elif mode == "pre":
                h = h + torch.tanh(norm(h)) @ W
            else:
                h = norm(h + torch.tanh(h) @ W)
            trace.append(h.pow(2).mean().sqrt().item())
        traces[mode] = trace

plt.figure(figsize=(7, 4))
for mode, color in (("none", "#BF616A"), ("pre", "#5E81AC"), ("post", "#A3BE8C")):
    plt.plot(range(1, layers + 1), traces[mode], label=mode, color=color)
plt.yscale("log"); plt.xlabel("layer"); plt.ylabel("stream rms")
plt.title("Nonlinear residual chain: 36 layers")
plt.legend(); plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

for mode, trace in traces.items():
    print(f"{mode:>5}: layer 1 = {trace[0]:8.3f} → layer 36 = {trace[-1]:10.3f}")

Read the first and last RMS values for each branch. Normalizing after the addition keeps RMS near one, subject to epsilon; the other branches can accumulate updates. The plot determines the size of that growth.

Next ask the question in a trained three-layer MiniLLM. Its initialization and sublayer function differ from the illustrative chain, so the earlier numbers are not predictions for the model. Compare loss and layerwise RMS under a shared budget and optimizer configuration.

In [ ]:
def stream_rms(model: MiniLLM) -> list[float]:
    ids = valid_stream.batch(torch.arange(4)).to(DEVICE)
    values = []
    with torch.no_grad():
        x = model.embed_tokens(ids)
        cos, sin = model.rope(torch.arange(ids.shape[1], device=DEVICE))
        for layer in model.layers:
            x = layer(x, cos, sin)
            values.append(x.float().pow(2).mean().sqrt().item())
    return values

print("norm placement ablation:")
norm_runs = [run(f"norm_placement={p}", norm_placement=p)
             for p in ("pre", "post", "peri", "olmo2", "none")]

print("\nresidual stream rms by layer of the trained models:")
for r in norm_runs:
    r["rms"] = stream_rms(r["model"])
    print(f"  {r['label']:<26} " +
          " → ".join(f"{v:.2f}" for v in r["rms"]))

Loss and residual RMS answer different questions. If a variant has higher loss, inspect its activation and gradient scales before diagnosing an exploding stream. Conversely, modest growth across three layers does not refute a depth-related problem.

Peri-LN normalizes both sides of a sublayer; the OLMo2-style graph normalizes its output before addition. Changing the update scale can change the appropriate learning rate. A shared rate is a useful control, not a complete tuning protocol. This table compares configurations; comparing methods requires per-method tuning and repeated runs.

## 3. MHA / GQA / MQA (M5)

Here scale is not in the way: the cache saving is a property of geometry rather
than of training, and it computes exactly. The open question is different — what
the model loses in quality.

In [ ]:
print("attention ablation:")
attn_runs = []
for name, kv in (("MHA (H_kv=4)", 4), ("GQA (H_kv=2)", 2), ("MQA (H_kv=1)", 1)):
    r = run(name, num_key_value_heads=kv)
    r["kv_per_token"] = kv_bytes_per_token(r["cfg"])
    attn_runs.append(r)

print(f"\n{'scheme':<14} {'KV, B/token':>12} {'× vs MHA':>9} {'valid loss':>11} {'Δloss':>8}")
mha = attn_runs[0]
for r in attn_runs:
    print(f"{r['label']:<14} {r['kv_per_token']:>12,} "
          f"{mha['kv_per_token'] / r['kv_per_token']:>8.1f}× "
          f"{r['loss']:>11.4f} {r['loss'] - mha['loss']:>+8.4f}")

With length, layer count, and dtype fixed, KV storage changes by `H/H_kv`. This follows from configuration geometry independently of training quality. The loss difference is empirical: read its sign and magnitude from this run.

A small gap does not establish that reducing KV heads is always free, and a large gap does not rank attention schemes for every budget. Projection counts and optimization trajectories also change. Broader conclusions need repeated runs and suitable tuning.

## 4. Activations within the GLU family (Module 6)

Compare SwiGLU, GeGLU, and ReGLU. Each uses two input matrices and an output matrix; the gate activation changes. Equal widths give equal parameter counts, which the assertion verifies.

The $d_{ff}\approx8d/3$ rule matches a three-matrix GLU budget to a conventional two-matrix FFN of width $4d$. We round the width for this implementation and do not implement the two-matrix control. The experiment therefore compares GLU activations, not the value of gating itself.

In [ ]:
d = BASE["hidden_size"]
gated_ff = round(8 * d / 3 / 64) * 64          # the 2/3 rule, aligned to 64
print(f"all variants: 3 · {d} · {gated_ff} = {3 * d * gated_ff:,} FFN parameters\n")

print("ablation of φ inside the GLU family:")
ffn_runs = [run(f"{name} (3 matrices)", hidden_act=act, intermediate_size=gated_ff)
            for name, act in (("SwiGLU", "swiglu"), ("GeGLU", "geglu"),
                              ("ReGLU", "relu"))]

assert len({r["params"] for r in ffn_runs}) == 1, "budgets must match"
best = min(ffn_runs, key=lambda r: r["loss"])
spread = max(r["loss"] for r in ffn_runs) - min(r["loss"] for r in ffn_runs)
print(f"\nbudgets match to the parameter: {ffn_runs[0]['params']:,}")
print(f"best on this run — {best['label']}, loss spread {spread:.4f}")

Check parameter equality and the observed loss spread. The code names the lowest-loss variant in this run, but one seed cannot estimate between-run variability. Repeat under several seeds with the same protocol and evaluation inputs before calling the difference stable.

Here `hidden_act="relu"` selects ReGLU: ReLU does not remove the second input matrix. Configuration names must be read alongside the computation graph. An ungated FFN remains outside this comparison.

## 5. Dense and MoE at similar active budgets (Module 7)

Keep vocabulary, attention, depth, and positions fixed while replacing the dense FFN with experts. Rounding expert width to a multiple of 64 can leave different active counts. The table reports that ratio alongside total parameters and loss; KV geometry stays shared.

Both branches receive twice the step budget of the other experiments. This matches processed tokens, not wall time or every FLOP. Routing and combining expert outputs add work. We observe this particular model pair, then investigate router load separately.

In [ ]:
from minillm.model import MoELayer

MOE_STEPS = STEPS * 2
EXPERTS, TOPK, SHARED = 8, 2, 1
expert_ff = max(64, round(gated_ff / (TOPK + SHARED) / 64) * 64)

print("sparsity ablation (step budget doubled):")
dense = run("dense", steps=MOE_STEPS, intermediate_size=gated_ff)
moe = run(f"MoE {EXPERTS} experts, top-{TOPK}", steps=MOE_STEPS,
          intermediate_size=gated_ff, num_local_experts=EXPERTS,
          num_experts_per_tok=TOPK, num_shared_experts=SHARED,
          moe_intermediate_size=expert_ff, first_k_dense_replace=1)

parity = moe["active"] / dense["active"]
sparsity = moe["params"] / moe["active"]
print(f"\n{'variant':<26} {'total':>12} {'active':>11} {'valid loss':>11}")
for r in (dense, moe):
    print(f"{r['label']:<26} {r['params']:>12,} {r['active']:>11,} {r['loss']:>11.4f}")
print(f"\nMoE active relative to dense: ×{parity:.3f} — active counts; runtime is measured separately")
print(f"capacity: total ×{moe['params'] / dense['params']:.2f}, "
      f"sparsity ×{sparsity:.2f}")
print(f"KV cache matches to the byte: "
      f"{kv_bytes_per_token(dense['cfg']) == kv_bytes_per_token(moe['cfg'])}")

In [ ]:
# What about expert load? This is the first thing to look at in an MoE.
def load_table(result, title):
    layers = [l.ffn for l in result["model"].layers if isinstance(l.ffn, MoELayer)]
    print(title)
    print(f"{'layer':>5} {'min':>7} {'max':>7} {'std':>7}  {'max|bias|':>9}")
    for i, layer in enumerate(layers, start=1):
        load = layer.expert_load
        print(f"{i:>5} {load.min():>7.3f} {load.max():>7.3f} {load.std():>7.3f}"
              f"  {layer.expert_bias.abs().max():>9.4f}")
    return max(float(l.expert_load.std()) for l in layers)

ideal = TOPK / EXPERTS
print(f"a perfectly even load would be {ideal:.3f} per expert\n")
std_default = load_table(moe, "default bias step (router_bias_speed = 1e-3):")

Inspect minimum, maximum, and standard deviation of load. This is a snapshot of the last measured batch, not a training-wide average. Concentrated assignments can signal imbalance; one snapshot cannot establish persistent collapse.

Increase the balancing-bias update rate from $10^{-3}$ to $5\cdot10^{-2}$ with other settings fixed. Compare both load and loss. A narrower load distribution need not improve predictions, because routing also has to learn which experts are useful.

In [ ]:
moe_fast = run(f"MoE, balance ×50", steps=MOE_STEPS, intermediate_size=gated_ff,
               num_local_experts=EXPERTS, num_experts_per_tok=TOPK,
               num_shared_experts=SHARED, moe_intermediate_size=expert_ff,
               first_k_dense_replace=1, router_bias_speed=5e-2)
print()
std_fast = load_table(moe_fast, "accelerated bias (router_bias_speed = 5e-2):")
print(f"\nmaximum load spread: {std_default:.3f} → {std_fast:.3f}")
print(f"valid loss: {moe['loss']:.4f} → {moe_fast['loss']:.4f} "
      f"(dense {dense['loss']:.4f})")

### Expert capacity: how much is lost silently

An expert's buffer is finite and the router does not know it. `capacity_factor`
sets the headroom above a perfectly even load; whatever does not fit is dropped.
The run below is forward-only on the trained model — training is left alone.

In [ ]:
probe = valid_stream.batch(torch.arange(8)).to(DEVICE)
print(f"{'capacity_factor':>16} {'ceiling':>9} {'dropped':>11}")
capacity_rows = []
for factor in (None, 2.0, 1.5, 1.25, 1.0, 0.75):
    cfg_c = MiniLLMConfig(**{**moe["cfg"].to_dict(), "capacity_factor": factor})
    probe_model = MiniLLM(cfg_c).to(DEVICE)
    probe_model.load_state_dict(moe["model"].state_dict())
    with torch.no_grad():
        probe_model(probe)
    layer = next(l.ffn for l in probe_model.layers if isinstance(l.ffn, MoELayer))
    cap = cfg_c.expert_capacity(probe.numel())
    capacity_rows.append(dict(factor=factor, capacity=cap, dropped=float(layer.dropped_fraction)))
    print(f"{str(factor):>16} {str(cap):>9} {layer.dropped_fraction.item():>10.1%}")

Total and active counts and KV equality follow from the configs; loss and load come from the experiment. Keep those kinds of evidence separate. If faster balancing narrows load spread without closing the dense-model gap, imbalance has not thereby been ruled out as a contributing cause. The intervention may be incomplete and may change optimization in other ways.

Uniform top-2 routing among eight experts would assign a quarter of tokens to each on average. That is a conditional expectation, not a guaranteed load or a count of optimizer steps. A short budget limits the experiment without automatically explaining every outcome.

The capacity table measures dropped assignments for fixed weights. A limit based on balanced load can overflow under unequal assignments. Dropping an assignment does not remove the whole token: residual paths, attention, and other experts remain. Evaluating its quality cost would require scoring each capacity setting; this table measures dropping itself.

## 6. RoPE beyond training lengths (Module 4)

A long-range test needs a task containing a long-range dependency. Train a separate model on a random sequence repeated twice. Targets in the second half can be recovered from the first, with half-length controlling dependency distance. Successful copying does not identify a particular induction head; that would require evidence about internal computations.

Training distances range from 12 to 48, and evaluation distances from 48 to 128. First measure accuracy at distance 32. If the model has not learned the in-range task, low out-of-range accuracy cannot confidently diagnose failed extrapolation. Uniform guessing gives $1/64=0.015625$. Fast validation uses only ten training steps and checks execution.


Sequence length varies across batches, so the budget is measured from actual input positions and supervised targets. The training config's `seq_len` does not control this generator. The next cell records both counts for each RoPE setting.

In [ ]:
from minillm.probes import RepeatTask, chance_level, repeat_accuracy
from minillm.model import RotaryEmbedding

PROBE_V = 64
probe = RepeatTask(vocab_size=PROBE_V, train_range=(12, 48), seed=SEED)
PROBE_BASE = dict(vocab_size=PROBE_V, hidden_size=128, num_hidden_layers=2,
                  num_attention_heads=4, num_key_value_heads=2, head_dim=32,
                  max_position_embeddings=512)
PROBE_STEPS = 1200 if DEVICE == "cpu" else 3000
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    PROBE_STEPS = 10

probe_budgets = {}

def train_probe(theta: float) -> MiniLLM:
    cfg = MiniLLMConfig.profile("S", **{**PROBE_BASE, "rope_theta": theta})
    torch.manual_seed(SEED)
    model = MiniLLM(cfg).to(DEVICE)
    probe_state = train(model, probe, TrainConfig(steps=PROBE_STEPS, seq_len=64,
                                    micro_batch_size=16, optimizer="muon",
                                    seed=SEED, device=DEVICE, log_every=10_000))
    probe_budgets[str(theta)] = {
        "input_positions": probe_state.tokens,
        "supervised_targets": probe_state.supervised_tokens,
    }
    return model

probe_models = {theta: train_probe(theta) for theta in (10_000.0, 500_000.0)}
print(f"chance level: {chance_level(probe):.4f}")
print("measured RoPE budgets:", probe_budgets)
probe_in_range = {str(theta): repeat_accuracy(model, probe, 32, device=DEVICE)
                  for theta, model in probe_models.items()}
for theta, accuracy in probe_in_range.items():
    print("rope_theta:", theta, "in-range accuracy at 32:", accuracy)

In [ ]:
LENGTHS = (48, 64, 80, 96, 128)
rope_rows = {f"base {theta:g}": [repeat_accuracy(m, probe, n, device=DEVICE)
                                 for n in LENGTHS]
             for theta, m in probe_models.items()}

# YaRN on top of an already trained model: RoPE has no parameters, so swapping
# the frequency table is enough — the weights stay as they are. That is the
# "stretching without fine-tuning" of M4. The pretraining window is 96 tokens
# (two halves of 48).
for factor in (2.0, 4.0):
    model = probe_models[10_000.0]
    saved = model.rope
    yarn_cfg = MiniLLMConfig.profile("S", **{**PROBE_BASE, "rope_theta": 10_000.0,
        "rope_scaling": {"rope_type": "yarn", "factor": factor,
                         "original_max_position_embeddings": 96}})
    model.rope = RotaryEmbedding(yarn_cfg).to(DEVICE)
    rope_rows[f"10000 + YaRN ×{factor:g}"] = [
        repeat_accuracy(model, probe, n, device=DEVICE) for n in LENGTHS]
    model.rope = saved

print(f"{'variant':<22}" + " ".join(f"{n:>8}" for n in LENGTHS))
print("-" * 64)
for name, values in rope_rows.items():
    print(f"{name:<22}" + " ".join(f"{v:>8.3f}" for v in values))
print("\ntraining used distances 12–48; everything right of 48 is extrapolation")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
palette = ("#BF616A", "#D08770", "#5E81AC", "#A3BE8C")
for (name, values), color in zip(rope_rows.items(), palette):
    plt.plot(LENGTHS, values, "o-", label=name, color=color)
plt.axvline(48, ls="--", lw=1, color="#4C566A")
plt.text(49, 0.5, "training boundary", rotation=90, va="center",
         fontsize=9, color="#4C566A")
plt.axhline(chance_level(probe), ls=":", lw=1, color="#4C566A")
plt.text(120, chance_level(probe) + 0.03, "chance level",
         ha="right", fontsize=9, color="#4C566A")
plt.xlabel("dependency distance (half length)")
plt.ylabel("share of correctly restored tokens")
plt.title('Accuracy across dependency distances')
plt.legend(); plt.ylim(-0.03, 1.05)
plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

Read the graph in two stages. Compare the two RoPE bases inside and beyond the training range. Then compare the original 10,000-base model with YaRN while holding learned weights fixed. The package's YaRN implementation defines the intervention, including any attention scaling specified by that mechanism.

Any long-range gain or short-range loss belongs to this training run and factor. Neither failure of ordinary RoPE nor recovery under YaRN is guaranteed. Retain every point, including in-range accuracy. Selecting a factor from these results turns them into development evidence and requires a fresh test.

## 7. Muon and AdamW: selection and held-out evaluation

Two default settings do not characterize the full potential of two optimizers. Search two learning rates for each under a common budget. Select on validation loss, then score the selected model on a separate test split kept out of selection.

The split precedes tokenizer training. All partitions share the synthetic generator, so test performance concerns new synthetic lines, not natural language. Repeating across seeds and widening the grid are separate follow-up experiments.

In [ ]:
def run_optimizer(optimizer: str, lr: float):
    cfg = MiniLLMConfig.profile("S", **BASE)
    torch.manual_seed(SEED)
    model = MiniLLM(cfg).to(DEVICE)
    tcfg = TrainConfig(steps=STEPS, seq_len=128, micro_batch_size=8,
                       optimizer=optimizer, lr=lr if optimizer == "adamw" else 3e-3,
                       muon_lr=lr if optimizer == "muon" else 0.02,
                       seed=SEED, device=DEVICE, log_every=10_000)
    train(model, train_stream, tcfg)
    return dict(model=model, config=tcfg, valid_loss=evaluate(model, valid_stream, tcfg, batches=8))

grid = {
    "adamw": [3e-3, 1e-2],
    "muon":  [2e-2, 6e-2],
}
optimizer_report = {}
for optimizer, lrs in grid.items():
    runs = {lr: run_optimizer(optimizer, lr) for lr in lrs}
    chosen_lr, selected = min(runs.items(), key=lambda kv: kv[1]["valid_loss"])
    test_loss = evaluate(selected["model"], test_stream, selected["config"], batches=8)
    optimizer_report[optimizer] = dict(selected_lr=chosen_lr,
        validation_grid={str(lr): r["valid_loss"] for lr,r in runs.items()},
        selected_valid_loss=selected["valid_loss"], selected_test_loss=test_loss)
    print(optimizer, optimizer_report[optimizer])

Compare selected learning rates, validation loss, and test loss. The test score did not participate in selection, making it a separate check of that choice. Two grid points do not guarantee good tuning; other optimizer coefficients remain fixed.

The run prints its horizon: 60 CPU steps or 600 GPU steps by default, reduced to 5 in fast validation. A short horizon can favor a quick starter without predicting long-run performance. One seed cannot establish a robust advantage; repeat the selection-and-test procedure to assess it.

## 8. From measurements to a saved report

Write the settings and numerical results to JSON, reload the file, and construct the summary from that saved record. The table can then be inspected independently of the training cells. Label geometry-derived quantities as calculations and loss, RMS, load, and accuracy as observations.

Some course questions are not tested here, such as packed-quantization speed or RL quality. Those receive neither invented values nor reproduction verdicts. Fast-mode reports explicitly identify their role as execution checks.

In [ ]:
CAPSTONE_LANGUAGE = 'en'
"""Embedded into both ablation notebooks; reads only this run's measurements."""
import json
from pathlib import Path

ablation_record = {
    "schema": 2,
    "language": CAPSTONE_LANGUAGE,
    "mode": "fast_execution_check" if os.getenv("CAPSTONE_FAST_VALIDATION") == "1" else "teaching_experiment",
    "seed": SEED, "device": DEVICE, "torch": torch.__version__,
    "steps": STEPS, "moe_steps": MOE_STEPS, "probe_steps": PROBE_STEPS,
    "data": {"train_sha": train_stream.checksum(), "valid_sha": valid_stream.checksum(),
             "test_sha": test_stream.checksum()},
    "chain_rms": traces,
    "norm": [{"label": r["label"], "valid_loss": r["loss"], "rms": r["rms"]} for r in norm_runs],
    "attention": [{"label": r["label"], "valid_loss": r["loss"],
                   "calculated_kv_bytes_per_token_bf16": r["kv_per_token"]} for r in attn_runs],
    "glu": [{"label": r["label"], "valid_loss": r["loss"], "parameters": r["params"]} for r in ffn_runs],
    "moe": {"dense_total": dense["params"], "moe_total": moe["params"],
            "dense_active": dense["active"], "moe_active": moe["active"],
            "active_ratio": parity, "dense_loss": dense["loss"], "moe_loss": moe["loss"],
            "faster_balance_loss": moe_fast["loss"],
            "last_batch_load_std": std_default, "faster_balance_load_std": std_fast},
    "capacity": capacity_rows,
    "rope": {"distances": list(LENGTHS), "accuracy": rope_rows,
             "training_budgets": probe_budgets,
             "in_range_32": probe_in_range, "chance": chance_level(probe)},
    "optimizers": optimizer_report,
    "not_tested": ["ungated FFN", "packed quantization speed", "RL answer quality",
                   "scaling-law optimum", "QK-norm ablation",
                   "speculative distribution test in this notebook", "EOT ablation in this notebook"],
}
report_dir = Path("runs/ablations")
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "measurements.json"
report_path.write_text(json.dumps(ablation_record, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
saved = json.loads(report_path.read_text(encoding="utf-8"))
assert saved == ablation_record
ru = CAPSTONE_LANGUAGE == "ru"
mode_ru = {"fast_execution_check": "быстрая проверка исполнения", "teaching_experiment": "учебный эксперимент"}
print("Режим:" if ru else "Mode:", mode_ru[saved["mode"]] if ru else saved["mode"])
print("Измерения сохранены:" if ru else "Measurements saved:", report_path)
labels_ru = {"chain_rms": "RMS остаточной цепочки", "norm": "Нормализация",
             "attention": "Внимание", "glu": "Варианты GLU", "moe": "Эксперты",
             "capacity": "Ограничение нагрузки", "rope": "Позиции", "optimizers": "Оптимизаторы"}
for key in ("chain_rms", "norm", "attention", "glu", "moe", "capacity", "rope", "optimizers"):
    print("\n" + (labels_ru[key] if ru else key))
    print(json.dumps(saved[key], ensure_ascii=False, indent=2))
print("\nЗдесь не проверялось:" if ru else "\nNot tested here:")
not_tested_ru = {"ungated FFN": "FFN без гейта",
    "packed quantization speed": "скорость упакованной квантизации",
    "RL answer quality": "качество ответов после RL",
    "scaling-law optimum": "оптимум закона масштабирования",
    "QK-norm ablation": "абляция QK-нормировки",
    "speculative distribution test in this notebook": "распределение спекулятивного семплирования в этой тетради",
    "EOT ablation in this notebook": "абляция EOT в этой тетради"}
for item in saved["not_tested"]:
    print("-", not_tested_ru[item] if ru else item)

## 9. Continuing the investigation

Choose one observation and pose a follow-up that could change your interpretation. Does the normalization loss gap survive per-mode learning-rate tuning? Does the MoE comparison change with the data distribution? Has the repetition task been learned before extrapolation is judged?

Retain the control, budget, data, and selection rule. Show variability when outcomes depend on the seed, and leave unmeasured quantities absent. A small or missing effect in one run limits that run's conclusion without dismissing the mechanism everywhere.

The project closes the course by connecting formulas, executable code, and evidence. Your report should let another reader distinguish measurements from config-derived calculations and identify the stronger claim that still needs a test.